# Tokenizer from scratch

This notebook builds a byte pair encoding (BPE) tokenizer in five steps, the way GPT-4 does it.

1. BasicTokenizer: train, encode, decode on raw UTF-8 bytes.
2. RegexTokenizer: split text with the GPT-4 pattern first, then run BPE inside each piece.
3. GPT4Tokenizer: load the real GPT-4 (cl100k_base) merges and match tiktoken exactly.
4. Special tokens such as <|endoftext|>.
5. Stretch goal: BPE on Unicode code points (SentencePiece style) and match the Llama 2 tokenizer.

Every step ends with tests. The tests compare against tiktoken and SentencePiece, so a passing notebook means the tokenizers really agree with the reference implementations.

Files used (downloaded automatically if missing): data/taylorswift.txt, data/input.txt (Shakespeare), data/llama2_tokenizer.model.

In [1]:
import os, time, heapq, random, json, urllib.request
from collections import Counter

import regex
import tiktoken

os.makedirs("data", exist_ok=True)
os.makedirs("tiktoken_cache", exist_ok=True)
# tiktoken normally downloads the GPT-4 vocabulary on first use. Pointing the cache at a local folder lets it be reused offline.
os.environ.setdefault("TIKTOKEN_CACHE_DIR", os.path.abspath("tiktoken_cache"))

def fetch(url, path):
    if not os.path.exists(path):
        print("downloading", path)
        urllib.request.urlretrieve(url, path)
    return path

fetch("https://raw.githubusercontent.com/karpathy/minbpe/master/tests/taylorswift.txt", "data/taylorswift.txt")
fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "data/input.txt")
fetch("https://raw.githubusercontent.com/karpathy/llama2.c/master/tokenizer.model", "data/llama2_tokenizer.model")

taylor = open("data/taylorswift.txt", encoding="utf-8").read()
print("taylorswift.txt:", len(taylor), "characters,", len(taylor.encode("utf-8")), "bytes")

taylorswift.txt: 185561 characters, 185768 bytes


## Step 1: BasicTokenizer

BPE starts from the 256 possible byte values. It repeatedly finds the most frequent adjacent pair of token ids, gives that pair a new id, and replaces every occurrence. After N merges the vocabulary has 256 + N tokens.

Two helper functions do the work: get_stats counts adjacent pairs, and merge replaces a pair with a new id.

In [2]:
def get_stats(ids, counts=None):
    """Count how often each adjacent pair occurs in a list of ids."""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    """Replace every occurrence of pair in ids with the new token idx."""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

class BasicTokenizer:
    def __init__(self):
        self.merges = {}                      # (id, id) -> new id, in the order they were learned
        self.vocab = self._build_vocab()      # id -> bytes

    def _build_vocab(self):
        vocab = {i: bytes([i]) for i in range(256)}
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]
        return vocab

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256
        ids = list(text.encode("utf-8"))
        merges = {}
        vocab = {i: bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)   # ties go to the pair seen first
            idx = 256 + i
            ids = merge(ids, pair, idx)
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i + 1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) had {stats[pair]} occurrences")
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            # apply the earliest learned merge that is present
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[i] for i in ids)
        return text_bytes.decode("utf-8", errors="replace")

A first check on a tiny example. For the text aaabdaaabac with three merges, the answer is known: (a,a) becomes 256, (256,a) becomes 257, (257,b) becomes 258, and the text becomes [258, 100, 258, 97, 99].

In [3]:
tok = BasicTokenizer()
tok.train("aaabdaaabac", 256 + 3, verbose=True)
print(tok.encode("aaabdaaabac"))
assert tok.encode("aaabdaaabac") == [258, 100, 258, 97, 99]
assert tok.decode([258, 100, 258, 97, 99]) == "aaabdaaabac"
print("ok")

merge 1/3: (97, 97) -> 256 (b'aa') had 4 occurrences
merge 2/3: (256, 97) -> 257 (b'aaa') had 2 occurrences
merge 3/3: (257, 98) -> 258 (b'aaab') had 2 occurrences
[258, 100, 258, 97, 99]
ok


Now train on the Taylor Swift text with a vocabulary of 512 (256 merges).

In [4]:
basic = BasicTokenizer()
t0 = time.time()
basic.train(taylor, 512)
print(f"trained in {time.time() - t0:.1f}s")

def show(b):
    return b.decode("utf-8", errors="replace")

print("first 15 merges")
for (p0, p1), idx in list(basic.merges.items())[:15]:
    print(f"  {idx}: {basic.vocab[p0]!r} + {basic.vocab[p1]!r} -> {basic.vocab[idx]!r}")
print("last 10 merges")
for (p0, p1), idx in list(basic.merges.items())[-10:]:
    print(f"  {idx}: {basic.vocab[idx]!r}")

ids = basic.encode(taylor)
print(f"compression: {len(taylor.encode('utf-8'))} bytes -> {len(ids)} tokens ({len(taylor.encode('utf-8')) / len(ids):.2f}x)")

trained in 5.1s
first 15 merges
  256: b'e' + b' ' -> b'e '
  257: b',' + b' ' -> b', '
  258: b'd' + b' ' -> b'd '
  259: b'.' + b' ' -> b'. '
  260: b'r' + b' ' -> b'r '
  261: b'2' + b'0' -> b'20'
  262: b's' + b' ' -> b's '
  263: b'i' + b'n' -> b'in'
  264: b'o' + b'n' -> b'on'
  265: b'r' + b'i' -> b'ri'
  266: b't' + b' ' -> b't '
  267: b't' + b'h' -> b'th'
  268: b'e' + b'd ' -> b'ed '
  269: b', ' + b'20' -> b', 20'
  270: b'a' + b'n' -> b'an'
last 10 merges
  502: b'An'
  503: b'18'
  504: b'for'
  505: b'ountry '
  506: b'Ame'
  507: b'Americ'
  508: b' T'
  509: b'sing'
  510: b'was '
  511: b'12'


compression: 185768 bytes -> 78746 tokens (2.36x)


To look at how a sentence is split, the helper below prints each token in its own colour.

In [5]:
from IPython.display import HTML, display

def visualize(tokenizer, text):
    colors = ["#cde7ff", "#ffe2b8", "#d5f5d0", "#f8d3e8", "#e6dcff"]
    spans = []
    ids = tokenizer.encode(text)
    for k, i in enumerate(ids):
        s = tokenizer.vocab[i].decode("utf-8", errors="replace")
        s = s.replace("&", "&amp;").replace("<", "&lt;").replace(" ", "&nbsp;").replace("\n", "\\n<br>")
        spans.append(f'<span style="background:{colors[k % len(colors)]};padding:1px;border:1px solid #bbb">{s}</span>')
    display(HTML("<div style='line-height:2'>" + "".join(spans) + "</div>"))
    print(len(ids), "tokens")

sample = "Taylor Swift released her album in 2014, and it sold 1,287,000 copies in the first week."
visualize(basic, sample)

41 tokens


The merges look reasonable: common pieces like " the", "ing" and ", " show up. But there is a problem. Tokens can cross categories, for example a letter followed by a comma and a space, or a number glued to a word. Step 2 fixes that.

Tests for step 1.

In [6]:
def make_test_strings():
    return [
        "",
        "a",
        "hello world",
        "hello world!!!? (안녕하세요!) lol123 😉",
        "   leading and trailing spaces   ",
        "line one\nline two\r\nline three\n\n\n",
        "tabs\tand\t\tmore",
        "I'll say we've seen it's what they're doing, he'd go, she's here, I'm fine",
        "1234567890 123 45 6",
        "def f(x):\n    return x ** 2  # square\n",
        "naïve café — “quotes” … ‘single’",
        "日本語のテキスト、中文文本、한국어 텍스트",
        "emoji 👩‍👩‍👧‍👦 🇨🇦 ❤️",
        "a" * 1000,
        " " * 50 + "x",
        "<|endoftext|> is just text here",
    ]

def fuzz_strings(n=200, seed=0):
    rng = random.Random(seed)
    ranges = [(32, 126), (0xA0, 0x24F), (0x400, 0x4FF), (0x3040, 0x30FF), (0xAC00, 0xAC80), (0x1F300, 0x1F64F), (9, 13)]
    out = []
    for _ in range(n):
        s = []
        for _ in range(rng.randint(0, 60)):
            lo, hi = rng.choice(ranges)
            s.append(chr(rng.randint(lo, hi)))
        out.append("".join(s))
    return out

def check_roundtrip(tokenizer, strings):
    for s in strings:
        ids = tokenizer.encode(s)
        assert tokenizer.decode(ids) == s, f"roundtrip failed for {s!r}"

check_roundtrip(basic, make_test_strings() + fuzz_strings())
print("BasicTokenizer: roundtrip ok on", len(make_test_strings()) + 200, "strings")

# encoding the training text should give the same ids that training produced
small = BasicTokenizer()
small.train(taylor[:5000], 300)
ids_a = small.encode(taylor[:5000])
assert small.decode(ids_a) == taylor[:5000]
# a bigger vocabulary can only make the encoding shorter, never longer
lengths = []
for v in (256, 270, 290, 320):
    t_v = BasicTokenizer(); t_v.train(taylor[:5000], v)
    lengths.append(len(t_v.encode(taylor[:5000])))
assert lengths == sorted(lengths, reverse=True) and lengths[0] > lengths[-1], lengths
print("encoded length by vocabulary size:", lengths)
# vocabulary that is bigger than the text allows should not crash
tiny = BasicTokenizer(); tiny.train("ab", 400)
assert tiny.decode(tiny.encode("abab")) == "abab"
print("BasicTokenizer: small-text and oversized vocabulary cases ok")

BasicTokenizer: roundtrip ok on 216 strings


encoded length by vocabulary size: [5006, 4205, 3706, 3303]
BasicTokenizer: small-text and oversized vocabulary cases ok


## Step 2: RegexTokenizer

GPT-2 and GPT-4 first split the text into pieces with a regular expression, then run BPE inside each piece. The pattern keeps letters, numbers, punctuation and whitespace apart, so no token can span two categories. The pieces are encoded separately and the results are joined.

Training also gets faster if identical pieces are counted once and weighted by how often they occur. This gives the same merges as processing every piece one by one, and a test below checks that.

In [7]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(BasicTokenizer):
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        super().__init__()
        self.pattern = pattern
        self.compiled = regex.compile(pattern)
        self._chunk_cache = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256
        counts = Counter(self.compiled.findall(text))          # piece -> how often it occurs
        words = [list(piece.encode("utf-8")) for piece in counts]
        freqs = list(counts.values())
        merges = {}
        vocab = {i: bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats = {}
            for w, f in zip(words, freqs):
                for pair in zip(w, w[1:]):
                    stats[pair] = stats.get(pair, 0) + f
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            words = [merge(w, pair, idx) if len(w) > 1 else w for w in words]
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i + 1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) had {stats[pair]} occurrences")
        self.merges = merges
        self.vocab = vocab
        self._chunk_cache = {}

    def _encode_chunk(self, chunk_bytes):
        cached = self._chunk_cache.get(chunk_bytes)
        if cached is not None:
            return cached
        ids = list(chunk_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        self._chunk_cache[chunk_bytes] = ids
        return ids

    def encode_ordinary(self, text):
        ids = []
        for piece in self.compiled.findall(text):
            ids.extend(self._encode_chunk(piece.encode("utf-8")))
        return ids

    def encode(self, text):
        return self.encode_ordinary(text)

In [8]:
rx = RegexTokenizer()
t0 = time.time()
rx.train(taylor, 512)
print(f"trained in {time.time() - t0:.1f}s")
print("first 15 merges")
for (p0, p1), idx in list(rx.merges.items())[:15]:
    print(f"  {idx}: {rx.vocab[p0]!r} + {rx.vocab[p1]!r} -> {rx.vocab[idx]!r}")

ids_rx = rx.encode(taylor)
n_bytes = len(taylor.encode("utf-8"))
print(f"compression: {n_bytes} bytes -> {len(ids_rx)} tokens ({n_bytes / len(ids_rx):.2f}x)")

visualize(basic, sample)
visualize(rx, sample)

trained in 2.0s
first 15 merges
  256: b'e' + b'r' -> b'er'
  257: b'2' + b'0' -> b'20'
  258: b'o' + b'r' -> b'or'
  259: b'i' + b'n' -> b'in'
  260: b'e' + b'd' -> b'ed'
  261: b' ' + b't' -> b' t'
  262: b'o' + b'n' -> b'on'
  263: b'h' + b'e' -> b'he'
  264: b' ' + b'S' -> b' S'
  265: b'a' + b'r' -> b'ar'
  266: b'a' + b'n' -> b'an'
  267: b' ' + b'A' -> b' A'
  268: b' t' + b'he' -> b' the'
  269: b'a' + b'l' -> b'al'
  270: b'r' + b'i' -> b'ri'
compression: 185768 bytes -> 87339 tokens (2.13x)


41 tokens


42 tokens


Now count how many learned tokens cross categories. A token crosses categories if the GPT-4 pattern would split its text into more than one piece, for example a word with trailing punctuation.

In [9]:
def crossing_tokens(tokenizer):
    bad = []
    for idx in range(256, max(tokenizer.vocab) + 1):
        try:
            s = tokenizer.vocab[idx].decode("utf-8")
        except UnicodeDecodeError:
            continue          # a token that is only part of a multi-byte character
        if len(regex.findall(GPT4_SPLIT_PATTERN, s)) != 1:
            bad.append(s)
    return bad

bad_basic = crossing_tokens(basic)
bad_rx = crossing_tokens(rx)
print("BasicTokenizer tokens that cross categories:", len(bad_basic), "of", len(basic.merges))
print("examples:", bad_basic[:12])
print("RegexTokenizer tokens that cross categories:", len(bad_rx), "of", len(rx.merges))
assert len(bad_rx) == 0
assert len(bad_basic) > 0

BasicTokenizer tokens that cross categories: 110 of 256
examples: ['e ', ', ', 'd ', '. ', 'r ', 's ', 't ', 'ed ', ', 20', 'er ', 'y ', 'the ']
RegexTokenizer tokens that cross categories: 0 of 256


Tests for step 2: roundtrip, the tiny known example, and a check that the weighted training gives the same merges as the plain piece by piece version.

In [10]:
def train_naive(pattern, text, vocab_size):
    """Reference version: keeps every piece as its own list, no counting trick."""
    chunks = [list(c.encode("utf-8")) for c in regex.findall(pattern, text)]
    merges = {}
    for i in range(vocab_size - 256):
        stats = {}
        for c in chunks:
            get_stats(c, stats)
        if not stats:
            break
        pair = max(stats, key=stats.get)
        chunks = [merge(c, pair, 256 + i) for c in chunks]
        merges[pair] = 256 + i
    return merges

sub = taylor[:30000]
fast = RegexTokenizer(); fast.train(sub, 300)
assert fast.merges == train_naive(GPT4_SPLIT_PATTERN, sub, 300)
print("weighted training matches the naive version: ok")

t = RegexTokenizer(); t.train("aaabdaaabac", 256 + 3)
assert t.encode("aaabdaaabac") == [258, 100, 258, 97, 99]

check_roundtrip(rx, make_test_strings() + fuzz_strings())
print("RegexTokenizer: roundtrip ok")

# no token id ever spans two regex pieces, so encoding pieces separately equals encoding the joined text
for s in make_test_strings()[:12]:
    parts = regex.findall(GPT4_SPLIT_PATTERN, s)
    assert "".join(parts) == s
print("pattern pieces always rejoin to the original text: ok")

weighted training matches the naive version: ok
RegexTokenizer: roundtrip ok
pattern pieces always rejoin to the original text: ok


## Step 3: Match GPT-4 and tiktoken

Load the real cl100k_base vocabulary with tiktoken. Two problems come up.

1. tiktoken stores the vocabulary as mergeable_ranks (bytes to rank), not as a list of merges. A merge can be recovered for each token by running BPE on its bytes using only ranks lower than its own: the result is always exactly two pieces, and those two pieces are the parents.
2. GPT-4 permutes the 256 raw byte values. The permutation is stored in the first 256 ranks. Bytes must be shuffled before encoding and unshuffled after decoding.

In [11]:
enc = tiktoken.get_encoding("cl100k_base")
mergeable_ranks = enc._mergeable_ranks
print("tokens with ranks:", len(mergeable_ranks), " special tokens:", enc._special_tokens)

def bpe_split(mergeable_ranks, token, max_rank):
    """Run BPE on the bytes of token, only allowing merges with rank below max_rank."""
    parts = [bytes([b]) for b in token]
    while True:
        best_i, best_rank = None, None
        for i in range(len(parts) - 1):
            rank = mergeable_ranks.get(parts[i] + parts[i + 1])
            if rank is not None and (best_rank is None or rank < best_rank):
                best_i, best_rank = i, rank
        if best_rank is None or best_rank >= max_rank:
            break
        parts = parts[:best_i] + [parts[best_i] + parts[best_i + 1]] + parts[best_i + 2:]
    return parts

def recover_merges(mergeable_ranks):
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue
        pair = bpe_split(mergeable_ranks, token, max_rank=rank)
        assert len(pair) == 2, (token, pair)
        merges[(mergeable_ranks[pair[0]], mergeable_ranks[pair[1]])] = rank
    return merges

t0 = time.time()
gpt4_merges = recover_merges(mergeable_ranks)
print(f"recovered {len(gpt4_merges)} merges in {time.time() - t0:.1f}s")

class GPT4Tokenizer(RegexTokenizer):
    def __init__(self, mergeable_ranks):
        super().__init__(GPT4_SPLIT_PATTERN)
        self.merges = recover_merges(mergeable_ranks)
        vocab = {i: bytes([i]) for i in range(256)}
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]
        self.vocab = vocab
        # the byte permutation lives in the first 256 ranks
        self.byte_shuffle = {i: mergeable_ranks[bytes([i])] for i in range(256)}
        self.inverse_byte_shuffle = {v: k for k, v in self.byte_shuffle.items()}

    def _encode_chunk(self, chunk_bytes):
        shuffled = bytes(self.byte_shuffle[b] for b in chunk_bytes)
        return super()._encode_chunk(shuffled)

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[i] for i in ids)
        text_bytes = bytes(self.inverse_byte_shuffle[b] for b in text_bytes)
        return text_bytes.decode("utf-8", errors="replace")

    def train(self, *args, **kwargs):
        raise NotImplementedError("GPT4Tokenizer is loaded, not trained")

gpt4 = GPT4Tokenizer(mergeable_ranks)

text = "hello world!!!? (안녕하세요!) lol123 😉"
mine = gpt4.encode(text)
ref = enc.encode(text)
print("mine:", mine)
print("tiktoken:", ref)
assert mine == ref
assert gpt4.decode(mine) == text
print("matches the example from the exercise")

tokens with ranks: 100256  special tokens: {'<|endoftext|>': 100257, '<|fim_prefix|>': 100258, '<|fim_middle|>': 100259, '<|fim_suffix|>': 100260, '<|endofprompt|>': 100276}


recovered 100000 merges in 0.6s


mine: [15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
tiktoken: [15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
matches the example from the exercise


The ranks are in the same order as the original byte shuffle, so a sentence like "hello world" gives the familiar ids 15339 and 1917. Tests: fixed strings, random strings, the whole Taylor Swift file, and random id sequences for decode.

In [12]:
def check_vs_tiktoken(tokenizer, strings):
    for s in strings:
        a = tokenizer.encode(s)
        b = enc.encode_ordinary(s)
        assert a == b, f"encode mismatch on {s!r}\n mine={a[:20]}\n ref ={b[:20]}"
        assert tokenizer.decode(a) == s

check_vs_tiktoken(gpt4, make_test_strings())
print("fixed strings: ok")

check_vs_tiktoken(gpt4, fuzz_strings(500, seed=1))
print("500 random unicode strings: ok")

t0 = time.time()
a = gpt4.encode(taylor)
print(f"encoded taylorswift.txt in {time.time() - t0:.1f}s ->", len(a), "tokens")
assert a == enc.encode_ordinary(taylor)
assert gpt4.decode(a) == taylor
print("taylorswift.txt identical to tiktoken: ok")

shakes = open("data/input.txt", encoding="utf-8").read()
assert gpt4.encode(shakes) == enc.encode_ordinary(shakes)
print("shakespeare identical to tiktoken: ok")

# decoding random id sequences, including ones that are invalid UTF-8
rng = random.Random(5)
for _ in range(300):
    ids = [rng.randrange(0, 100256) for _ in range(rng.randint(0, 30))]
    assert gpt4.decode(ids) == enc.decode(ids), ids
print("decode of random id sequences: ok")

fixed strings: ok
500 random unicode strings: ok
encoded taylorswift.txt in 0.1s -> 49298 tokens
taylorswift.txt identical to tiktoken: ok


shakespeare identical to tiktoken: ok
decode of random id sequences: ok


## Step 4: Special tokens

Special tokens like <|endoftext|> are never produced by BPE. The text is split on the special tokens first, each special token gets its fixed id, and the text between them is tokenized normally. To match tiktoken, the same rules apply:

- allowed_special="all" or a set: those tokens are encoded as special ids.
- disallowed_special="all" (the default): if the text contains a special token that is not allowed, raise an error.
- disallowed_special=(): such text is treated as ordinary characters.

During decoding, special ids are written out as plain text and are not passed through the byte shuffle.

In [13]:
class GPT4TokenizerSpecial(GPT4Tokenizer):
    def __init__(self, mergeable_ranks, special_tokens):
        super().__init__(mergeable_ranks)
        self.special_tokens = dict(special_tokens)
        self.inverse_special_tokens = {v: k for k, v in self.special_tokens.items()}

    def encode(self, text, allowed_special=set(), disallowed_special="all"):
        all_special = set(self.special_tokens)
        allowed = all_special if allowed_special == "all" else set(allowed_special)
        disallowed = all_special - allowed if disallowed_special == "all" else set(disallowed_special)
        for tok in disallowed:
            if tok in text:
                raise ValueError(f"text contains the special token {tok!r}, which is not allowed here")
        allowed = {k: v for k, v in self.special_tokens.items() if k in allowed}
        if not allowed:
            return self.encode_ordinary(text)
        pattern = "(" + "|".join(regex.escape(k) for k in allowed) + ")"
        ids = []
        for part in regex.split(pattern, text):
            if part in allowed:
                ids.append(allowed[part])
            elif part:
                ids.extend(self.encode_ordinary(part))
        return ids

    def decode(self, ids):
        out = bytearray()
        pending = bytearray()          # ordinary bytes still in the shuffled space
        for i in ids:
            if i in self.inverse_special_tokens:
                out += bytes(self.inverse_byte_shuffle[b] for b in pending)
                pending = bytearray()
                out += self.inverse_special_tokens[i].encode("utf-8")
            else:
                pending += self.vocab[i]
        out += bytes(self.inverse_byte_shuffle[b] for b in pending)
        return bytes(out).decode("utf-8", errors="replace")

gpt4s = GPT4TokenizerSpecial(mergeable_ranks, enc._special_tokens)
ids = gpt4s.encode("<|endoftext|>hello world", allowed_special="all")
print(ids, enc.encode("<|endoftext|>hello world", allowed_special="all"))
assert ids == enc.encode("<|endoftext|>hello world", allowed_special="all")
assert gpt4s.decode(ids) == "<|endoftext|>hello world"

[100257, 15339, 1917] [100257, 15339, 1917]


Tests: many combinations of allowed and disallowed settings, including the error cases, must behave exactly like tiktoken.

In [14]:
specials = list(enc._special_tokens)
cases = [
    "<|endoftext|>hello world",
    "hello<|endoftext|>",
    "<|endoftext|>",
    "a<|endoftext|><|endoftext|>b",
    "<|fim_prefix|>def f():<|fim_suffix|>return<|fim_middle|>x",
    "x <|endofprompt|> y <|endoftext|> z",
    "no special tokens here",
    "<|endoftext|",
    "almost <|endoftext|> but <|notatoken|> not",
    "한국어<|endoftext|>😉",
    "",
]

def outcome(fn):
    try:
        return ("ok", fn())
    except ValueError:
        return ("error", None)

settings = [
    dict(),
    dict(allowed_special="all"),
    dict(allowed_special={"<|endoftext|>"}),
    dict(allowed_special={"<|endoftext|>", "<|fim_prefix|>"}),
    dict(disallowed_special=()),
    dict(allowed_special="all", disallowed_special=()),
    dict(allowed_special={"<|endoftext|>"}, disallowed_special=()),
    dict(allowed_special=set(), disallowed_special=("<|endoftext|>",)),
]
n_checked = n_errors = 0
for text in cases:
    for kw in settings:
        a = outcome(lambda: gpt4s.encode(text, **kw))
        b = outcome(lambda: enc.encode(text, **kw))
        assert a == b, (text, kw, a, b)
        n_checked += 1
        n_errors += a[0] == "error"
        if a[0] == "ok":
            # decode must return the original text whenever specials were allowed or treated as text
            assert gpt4s.decode(a[1]) == enc.decode(b[1])
print(f"{n_checked} combinations match tiktoken ({n_errors} of them raise errors in both)")

# decode of special ids and mixed sequences
for ids in ([100257], [15339, 100257, 1917], [100276, 100258, 9906], []):
    assert gpt4s.decode(ids) == enc.decode(ids)
print("decode with special ids: ok")

# special tokens do not disturb ordinary text
check_vs_tiktoken(gpt4s, make_test_strings()[:10])
print("ordinary text through the special-token tokenizer: ok")

88 combinations match tiktoken (19 of them raise errors in both)
decode with special ids: ok
ordinary text through the special-token tokenizer: ok


## Step 5 (stretch): BPE on Unicode code points, matching Llama 2

Llama and Mistral use SentencePiece. The main differences from GPT-4:

- BPE runs on Unicode code points, not on UTF-8 bytes. Characters that are not in the vocabulary fall back to byte tokens like <0xF0>.
- Spaces are replaced by the symbol ▁, and a dummy ▁ is added at the start of the text.
- There is no regex pre-split. Instead of merge order, each piece has a score, and the merge loop always takes the adjacent pair whose merged piece has the highest score (ties go to the leftmost pair).
- Digits are always single tokens.

First, a look at what the Llama 2 model file says about itself.

In [15]:
import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as pb

sp = spm.SentencePieceProcessor(model_file="data/llama2_tokenizer.model")
proto = pb.ModelProto()
proto.ParseFromString(open("data/llama2_tokenizer.model", "rb").read())
ns, ts = proto.normalizer_spec, proto.trainer_spec
print("vocab size:", len(proto.pieces))
print("model type (2 = BPE):", ts.model_type, " byte_fallback:", ts.byte_fallback, " split_digits:", ts.split_digits)
print("normalizer:", repr(ns.name), " add_dummy_prefix:", ns.add_dummy_prefix, " remove_extra_whitespaces:", ns.remove_extra_whitespaces)
print("official encode:", sp.encode("hello world!!!? (안녕하세요!) lol123 😉", out_type=str))

vocab size: 32000
model type (2 = BPE): 2  byte_fallback: True  split_digits: True
normalizer: 'identity'  add_dummy_prefix: True  remove_extra_whitespaces: False
official encode: ['▁hello', '▁world', '!!!', '?', '▁(', '안', '<0xEB>', '<0x85>', '<0x95>', '하', '세', '요', '!)', '▁l', 'ol', '1', '2', '3', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x89>']


The class below can do two things: load any SentencePiece BPE model (so it can be checked against the official library), and train its own model from raw text on code points.

In [16]:
SPIECE = "▁"      # the visible space symbol used by SentencePiece
BYTE_TOKEN = regex.compile(r"^<0x([0-9A-F]{2})>$")

class CodePointBPE:
    def __init__(self, pieces, scores, kinds, add_dummy_prefix=True, remove_extra_whitespaces=False):
        # kinds: "normal", "unk", "control", "byte" for each piece
        self.pieces, self.scores, self.kinds = pieces, scores, kinds
        self.add_dummy_prefix = add_dummy_prefix
        self.remove_extra_whitespaces = remove_extra_whitespaces
        self.piece_to_id = {p: i for i, (p, k) in enumerate(zip(pieces, kinds)) if k == "normal"}
        self.byte_to_id, self.id_to_byte = {}, {}
        for i, (p, k) in enumerate(zip(pieces, kinds)):
            if k == "byte":
                b = int(BYTE_TOKEN.match(p).group(1), 16)
                self.byte_to_id[b] = i
                self.id_to_byte[i] = b
        self.unk_id = kinds.index("unk")

    @property
    def vocab_size(self):
        return len(self.pieces)

    # loading --------------------------------------------------------------
    @classmethod
    def from_sentencepiece(cls, path):
        m = pb.ModelProto()
        m.ParseFromString(open(path, "rb").read())
        assert m.trainer_spec.model_type == pb.TrainerSpec.BPE
        assert m.normalizer_spec.name in ("identity", ""), "only identity normalization is implemented"
        assert m.normalizer_spec.escape_whitespaces
        names = {1: "normal", 2: "unk", 3: "control", 4: "normal", 6: "byte"}     # user defined behaves like normal
        kinds = [names[p.type] for p in m.pieces]
        return cls([p.piece for p in m.pieces], [p.score for p in m.pieces], kinds,
                   m.normalizer_spec.add_dummy_prefix, m.normalizer_spec.remove_extra_whitespaces)

    # encoding -------------------------------------------------------------
    def _normalize(self, text):
        if self.remove_extra_whitespaces:
            text = " ".join(text.split(" ")).strip(" ")
            text = regex.sub(" +", " ", text)
        if self.add_dummy_prefix and text:
            text = " " + text
        return text.replace(" ", SPIECE)

    def _segment(self, s):
        """Greedy BPE on code points: always merge the adjacent pair with the highest score, leftmost first."""
        syms = list(s)
        n = len(syms)
        if n == 0:
            return []
        prev = list(range(-1, n - 1))
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        heap = []

        def push(l, r):
            if l == -1 or r == -1:
                return
            cand = syms[l] + syms[r]
            pid = self.piece_to_id.get(cand)
            if pid is not None:
                heapq.heappush(heap, (-self.scores[pid], l, len(cand), r))

        for i in range(n - 1):
            push(i, i + 1)
        while heap:
            _, l, size, r = heapq.heappop(heap)
            if syms[l] is None or syms[r] is None or len(syms[l]) + len(syms[r]) != size:
                continue                      # an out of date candidate
            syms[l] = syms[l] + syms[r]
            syms[r] = None
            nxt[l] = nxt[r]
            if nxt[r] != -1:
                prev[nxt[r]] = l
            push(prev[l], l)
            push(l, nxt[l])
        out, i = [], 0
        while i != -1:
            out.append(syms[i])
            i = nxt[i]
        return out

    def encode(self, text):
        ids = []
        for sym in self._segment(self._normalize(text)):
            pid = self.piece_to_id.get(sym)
            if pid is not None:
                ids.append(pid)
            elif self.byte_to_id:
                ids.extend(self.byte_to_id[b] for b in sym.encode("utf-8"))      # byte fallback
            else:
                ids.append(self.unk_id)
        return ids

    # decoding -------------------------------------------------------------
    def decode(self, ids):
        out = bytearray()
        first = True
        for i in ids:
            kind = self.kinds[i]
            if kind == "control":
                continue
            if kind == "byte":
                out.append(self.id_to_byte[i])
            else:
                piece = " ⁇ " if kind == "unk" else self.pieces[i]
                if first and self.add_dummy_prefix and piece.startswith(SPIECE):
                    piece = piece[1:]
                out += piece.replace(SPIECE, " ").encode("utf-8")
            first = False
        return out.decode("utf-8", errors="replace")

    # training -------------------------------------------------------------
    @classmethod
    def train(cls, text, vocab_size, add_dummy_prefix=True, verbose=False):
        norm = (" " + text if add_dummy_prefix and text else text).replace(" ", SPIECE)
        # words start at a space symbol; digits are always alone
        chunk_re = regex.compile(r"▁*[^▁\d]+|▁+|\d")
        counts = Counter(chunk_re.findall(norm))
        words = [list(c) for c in counts]
        freqs = list(counts.values())
        alphabet = Counter()
        for c, f in counts.items():
            for ch in c:
                alphabet[ch] += f
        pieces = ["<unk>", "<s>", "</s>"] + [f"<0x{b:02X}>" for b in range(256)]
        kinds = ["unk", "control", "control"] + ["byte"] * 256
        scores = [0.0] * len(pieces)
        for ch, _ in alphabet.most_common():
            pieces.append(ch); kinds.append("normal"); scores.append(0.0)
        seen = set(pieces)
        rank = 0
        while len(pieces) < vocab_size:
            stats = {}
            for w, f in zip(words, freqs):
                for pair in zip(w, w[1:]):
                    stats[pair] = stats.get(pair, 0) + f
            if not stats:
                break
            pair = max(stats, key=stats.get)
            new = pair[0] + pair[1]
            # merge inside all words
            for k, w in enumerate(words):
                if len(w) < 2:
                    continue
                out, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == pair[0] and w[i + 1] == pair[1]:
                        out.append(new); i += 2
                    else:
                        out.append(w[i]); i += 1
                words[k] = out
            if new not in seen:
                rank += 1
                pieces.append(new); kinds.append("normal"); scores.append(-float(rank))
                seen.add(new)
                if verbose:
                    print(f"piece {len(pieces) - 1}: {new!r} ({stats[pair]} occurrences)")
        return cls(pieces, scores, kinds, add_dummy_prefix, False)

Compare against the official Llama 2 tokenizer on many inputs. The check covers plain text, digits, newlines, repeated spaces, accented letters, Chinese and Korean text, emoji that are not in the vocabulary (byte fallback), empty strings, and random strings.

In [17]:
llama = CodePointBPE.from_sentencepiece("data/llama2_tokenizer.model")
print("loaded", llama.vocab_size, "pieces")

ex = "hello world!!!? (안녕하세요!) lol123 😉"
print("mine:    ", llama.encode(ex))
print("official:", sp.encode(ex))
assert llama.encode(ex) == sp.encode(ex)

llama_cases = make_test_strings() + [
    " ", "  ", "\n", " \n ", "\t", "▁", "a▁b", "x  y", " hello", "hello ", "I'm 25 years old, born in 1999.",
    "The quick brown fox jumps over the lazy dog. " * 20,
    "Ünïcödé tëxt with àccents and ñ", "Привет, мир!", "العربية", "שלום", "😀😃😄", "\x00\x01\x02", "tab\there",
]
for s in llama_cases:
    a, b = llama.encode(s), sp.encode(s)
    assert a == b, f"encode mismatch on {s!r}\n mine={a[:25]}\n ref ={b[:25]}"
    assert llama.decode(a) == sp.decode(b), f"decode mismatch on {s!r}"
print(len(llama_cases), "fixed cases: encode and decode match the official tokenizer")

for s in fuzz_strings(500, seed=2):
    a, b = llama.encode(s), sp.encode(s)
    assert a == b, f"fuzz mismatch on {s!r}"
    assert llama.decode(a) == sp.decode(b)
print("500 random strings: ok")

t0 = time.time()
a = llama.encode(taylor)
print(f"encoded taylorswift.txt in {time.time() - t0:.1f}s ->", len(a), "tokens")
assert a == sp.encode(taylor)
assert llama.decode(a) == sp.decode(a)
a = llama.encode(shakes)
assert a == sp.encode(shakes)
print("taylorswift.txt and shakespeare identical to the official Llama 2 tokenizer: ok")

loaded 32000 pieces
mine:     [22172, 3186, 21004, 29973, 313, 31734, 238, 136, 152, 30944, 31578, 31527, 14366, 301, 324, 29896, 29906, 29941, 29871, 243, 162, 155, 140]
official: [22172, 3186, 21004, 29973, 313, 31734, 238, 136, 152, 30944, 31578, 31527, 14366, 301, 324, 29896, 29906, 29941, 29871, 243, 162, 155, 140]
35 fixed cases: encode and decode match the official tokenizer
500 random strings: ok


encoded taylorswift.txt in 0.5s -> 63450 tokens


taylorswift.txt and shakespeare identical to the official Llama 2 tokenizer: ok


Digits are separate and unknown characters fall back to bytes, which is visible in the pieces below.

In [18]:
print([llama.pieces[i] for i in llama.encode("lol123 😉")])
print([llama.pieces[i] for i in llama.encode("Hello world")])

['▁l', 'ol', '1', '2', '3', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x89>']
['▁Hello', '▁world']


Now train a code point BPE of our own on the Taylor Swift text, and also train SentencePiece itself with the same settings as Llama (BPE, byte fallback, split digits, identity normalization). Our encoder must agree with the official library on a model that SentencePiece trained, and our own trained model must roundtrip.

In [19]:
import tempfile
sp_path = os.path.join(tempfile.mkdtemp(), "ts")
open(sp_path + ".txt", "w", encoding="utf-8").write(taylor)
spm.SentencePieceTrainer.train(
    input=sp_path + ".txt", model_prefix=sp_path, vocab_size=1000, model_type="bpe",
    byte_fallback=True, split_digits=True, normalization_rule_name="identity",
    remove_extra_whitespaces=False, add_dummy_prefix=True, character_coverage=1.0,
    pad_id=-1, unk_id=0, bos_id=1, eos_id=2, minloglevel=2, train_extremely_large_corpus=False)
sp_small = spm.SentencePieceProcessor(model_file=sp_path + ".model")
mine_sp = CodePointBPE.from_sentencepiece(sp_path + ".model")

probe = [taylor[:20000], shakes[:20000]] + llama_cases + fuzz_strings(300, seed=3)
for s in probe:
    assert mine_sp.encode(s) == sp_small.encode(s), f"mismatch on {s[:40]!r}"
    assert mine_sp.decode(mine_sp.encode(s)) == sp_small.decode(sp_small.encode(s))
print("encoder agrees with a freshly trained SentencePiece model on", len(probe), "inputs")

t0 = time.time()
own = CodePointBPE.train(taylor, 1000)
print(f"trained our own code point BPE in {time.time() - t0:.1f}s, vocab {own.vocab_size}")
print("learned pieces:", [p for p in own.pieces[-12:]])
for s in make_test_strings() + fuzz_strings(200, seed=4):
    assert own.decode(own.encode(s)) == s, f"roundtrip failed on {s!r}"
print("own model: roundtrip ok, including characters it has never seen (byte fallback)")

official = set(sp_small.id_to_piece(i) for i in range(sp_small.vocab_size()))
ours = set(own.pieces)
print(f"pieces shared with SentencePiece's own trainer: {len(ours & official)} of {len(ours)}")

encoder agrees with a freshly trained SentencePiece model on 337 inputs


W0000 00:00:1794623933.863456     139 spec_parser.h:262] --train_extremely_large_corpus is deprecated and ignored.


trained our own code point BPE in 5.3s, vocab 1000
learned pieces: ['esp', 'ake', '▁Midnight', '▁film', 'amed', 'ern', 'ales', '▁includ', '▁Sc', '▁ab', 'ey', '▁Br']
own model: roundtrip ok, including characters it has never seen (byte fallback)
pieces shared with SentencePiece's own trainer: 913 of 1000


## Wrap up

Save the Shakespeare corpus tokenized with a small RegexTokenizer so the GPT notebook can train on BPE tokens instead of characters. The GPT notebook reads data/shakespeare_bpe.npz.

In [20]:
import numpy as np
shake_tok = RegexTokenizer()
t0 = time.time()
shake_tok.train(shakes, 512)
print(f"trained BPE-512 on Shakespeare in {time.time() - t0:.1f}s")
shake_ids = np.array(shake_tok.encode(shakes), dtype=np.uint16)
assert shake_tok.decode(shake_ids.tolist()) == shakes
print(len(shakes), "characters ->", len(shake_ids), "tokens")
vocab_bytes = [shake_tok.vocab[i].decode("latin-1") for i in range(len(shake_tok.vocab))]
np.savez("data/shakespeare_bpe.npz", ids=shake_ids, vocab=np.array(vocab_bytes, dtype=object))
print("saved data/shakespeare_bpe.npz")

trained BPE-512 on Shakespeare in 5.4s


1115394 characters -> 547276 tokens
saved data/shakespeare_bpe.npz


What was built and checked:

- BasicTokenizer and RegexTokenizer with train, encode and decode, roundtrip tested on fixed and random strings.
- The regex split removes tokens that cross categories (checked by counting).
- GPT4Tokenizer gives the same ids as tiktoken on every test text, including two full books and random Unicode, and decodes random id sequences the same way.
- Special tokens behave like tiktoken for every combination tested, including the error cases.
- CodePointBPE reproduces the Llama 2 tokenizer and a freshly trained SentencePiece model exactly, and can train its own vocabulary.